# SNPデータ前処理

複数バッチの Genotyping TXT を読み込み、feather で保存する。

1. 各バッチを読み込み・転置し、共通SNPで結合
2. 欠損処理（高欠損の行・列、定数列を除外）
3. 数値変換（AA:0, AB:1, BB:2）・MAF/HWEフィルタ・平均補完
4. メタデータ順にSNP列を並び替え
5. feather 形式で保存（補完版・非補完版）

前処理アルゴリズムの本体は共有パッケージ `jbrt.snp`。

In [1]:
import json
import logging
from datetime import datetime

import pandas as pd

from jbrt import config, snp

# jbrt.snp の INFO ログ（各フィルタの除外件数）をノート上に表示する
_logger = logging.getLogger("jbrt")
_logger.setLevel(logging.INFO)
if not _logger.handlers:
    _h = logging.StreamHandler()
    _h.setFormatter(logging.Formatter("%(message)s"))
    _logger.addHandler(_h)
_logger.propagate = False

## 設定

In [ ]:
RAW_DIR, PROCESSED_DIR = config.RAW_DIR, config.PROCESSED_DIR

# 入力: SNP ディレクトリ配下を再帰的に探し、このパターンに合う TXT を全て使う
INPUT_DIR = RAW_DIR / "SNP"
TXT_PATTERN = "Genotyping_AX*.txt"

# 入力: アレイ注釈CSV
META_PATH = RAW_DIR / "SNP_meta" / "Axiom_BovMDv3.na35.r1.a1.annot.csv"


# フィルタ閾値
MISSING_THRESHOLD = 0.05   # 欠損率がこの値以上の行・列を除外
MAF_THRESHOLD = 0.01       # MAFがこの値未満のSNPを除外
HWE_THRESHOLD = 1e-4       # HWE検定のp値がこの値未満のSNPを除外

INPUT_DIR

PosixPath('/home/s_mori/JBRT/JBRT_data/raw/SNP')

## 1. 読み込み・結合

各バッチを `サンプル×SNP` に転置し、共通SNP（列の積集合）で縦結合する。

In [3]:
txt_paths = sorted(INPUT_DIR.rglob(TXT_PATTERN))
assert txt_paths, f"TXTが見つかりません: {INPUT_DIR} (pattern={TXT_PATTERN})"
print(f"対象TXT: {len(txt_paths)}件")

dfs = []
for p in txt_paths:
    d = snp.load_genotyping_txt(p)
    print(f"{p.name}: {d.shape[0]} サンプル × {d.shape[1]} SNP")
    dfs.append(d)

df = snp.concat_common_snps(dfs)
print(f"結合後: {df.shape[0]} サンプル × {df.shape[1]} SNP")
df.iloc[:3, :5]

対象TXT: 9件
Genotyping_AX003.txt: 96 サンプル × 57919 SNP
Genotyping_AX004.txt: 48 サンプル × 56854 SNP
Genotyping_AX020.txt: 669 サンプル × 58995 SNP
Genotyping_AX032.txt: 94 サンプル × 54465 SNP
Genotyping_AX034.txt: 96 サンプル × 58139 SNP
Genotyping_AX037.txt: 95 サンプル × 57864 SNP
Genotyping_AX041.txt: 95 サンプル × 53228 SNP
Genotyping_AX045.txt: 96 サンプル × 56917 SNP
Genotyping_AX050.txt: 96 サンプル × 59078 SNP
結合後: 1385 サンプル × 45098 SNP


probeset_id,AX-105094887,AX-106719429,AX-106719435,AX-106719437,AX-106719440
sample_id,,,,,
AX003_A01.CEL,AA,BB,BB,BB,AA
AX003_A03.CEL,AA,BB,BB,BB,AA
AX003_A05.CEL,AA,BB,BB,BB,AA


## 2〜4. 欠損処理・数値変換・並び替え

無効値除去 → 数値変換（AA:0, AB:1, BB:2）→ MAF/HWE フィルタ → 平均補完 → メタデータ順の並び替え 

In [4]:
# ============================================================
# 2. 欠損処理
# ============================================================
print("---- 2. 欠損処理 ----")
df = snp.filter_missing(df, threshold=MISSING_THRESHOLD)
print(f"処理後の形状: {df.shape}")

# ============================================================
# 3. 数値変換・MAF/HWE フィルタ・補完
# ============================================================
print("\n---- 3. 数値変換・MAF/HWE フィルタ・補完 ----")
df_numeric = snp.to_numeric(df)
df_filtered = snp.filter_maf(df_numeric, threshold=MAF_THRESHOLD)
df_filtered = snp.filter_hwe(df_filtered, threshold=HWE_THRESHOLD)

df_not_imputed = df_filtered.copy()
df_imputed = snp.mean_impute(df_filtered)
print(f"フィルタ後の形状: {df_filtered.shape}")
print(f"補完後の欠損数: {df_imputed.isna().sum().sum()}")

# ============================================================
# 4. メタデータ読み込み・並び替え
# ============================================================
print("\n---- 4. メタデータ読み込み・並び替え ----")
df_meta = snp.load_metadata(META_PATH)
print(f"メタデータ形状: {df_meta.shape}")

df_imputed = snp.reorder_by_metadata(df_imputed, df_meta)
df_not_imputed = snp.reorder_by_metadata(df_not_imputed, df_meta)
print(f"並び替え後の形状: {df_imputed.shape}")

---- 2. 欠損処理 ----


欠損処理: 欠損列除外=1328, 欠損行除外=0, 定数列除外=12621 (閾値=5%)


処理後の形状: (1385, 31149)

---- 3. 数値変換・MAF/HWE フィルタ・補完 ----


MAFフィルタ: 5036列除外 (MAF < 0.01)
HWEフィルタ: 432列除外 (p < 0.0001)


フィルタ後の形状: (1385, 25681)
補完後の欠損数: 0

---- 4. メタデータ読み込み・並び替え ----
メタデータ形状: (65008, 40)
並び替え後の形状: (1385, 25681)


## 5. 保存

`processed/snp_<YYYYMMDD>/` フォルダを作り、その中に保存する:

- `snp_imputed.feather` … 補完版
- `snp_not_imputed.feather` … 非補完版
- `manifest.json` … 入力TXT一覧・件数・閾値など（再現用）

In [ ]:
# 保存先: processed/snp_<YYYYMMDD>/
now = datetime.now()
run_dir = PROCESSED_DIR / f"snp_{now:%Y%m%d}"
run_dir.mkdir(parents=True, exist_ok=True)

out_imputed = run_dir / "snp_imputed.feather"
out_not_imputed = run_dir / "snp_not_imputed.feather"
out_manifest = run_dir / "manifest.json"

# sample_id を列に戻してから保存
df_imputed.reset_index().to_feather(out_imputed)
df_not_imputed.reset_index().to_feather(out_not_imputed)

# 再現用JSON
manifest = {
    "created": now.isoformat(timespec="seconds"),
    "n_txt": len(txt_paths),
    "txt_files": [str(p) for p in txt_paths],
    "n_samples": int(len(df_imputed)),
    "n_snps": int(df_imputed.shape[1]),
    "thresholds": {
        "missing": MISSING_THRESHOLD,
        "maf": MAF_THRESHOLD,
        "hwe": HWE_THRESHOLD,
    },
}
out_manifest.write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"保存先: {run_dir}")
print(f"  {out_imputed.name}")
print(f"  {out_not_imputed.name}")
print(f"  {out_manifest.name}")
print(f"サンプル数: {len(df_imputed)}, SNP数: {df_imputed.shape[1]}")

保存先: /home/s_mori/JBRT/JBRT_data/processed/snp_20260703
  snp_imputed.feather
  snp_not_imputed.feather
  manifest.json
サンプル数: 1385, SNP数: 25681
